# Assignment #1: The Donut Effect for Philadelphia ZIP Codes

## 1. Load the data

In [ ]:
import pandas as pd

zhvi = pd.read_csv("data/zillow_zhvi.csv")
zhvi.head()

## 2. Trim the data to just Philadelphia

In [ ]:
philly = zhvi.loc[(zhvi["City"] == "Philadelphia") & (zhvi["State"] == "PA")].copy()
philly.shape

## 3. Melt the data into tidy format

In [ ]:
id_columns = ["RegionID", "SizeRank", "RegionName", "RegionType",
              "StateName", "State", "City", "Metro", "CountyName"]
date_columns = [c for c in philly.columns if c not in id_columns]

philly_tidy = philly.melt(
    id_vars=["RegionName"],
    value_vars=date_columns,
    var_name="Date",
    value_name="ZHVI",
)
philly_tidy["Date"] = pd.to_datetime(philly_tidy["Date"])
philly_tidy.head()

## 4. Split the data for ZIP codes in/outside Center City

In [ ]:
greater_center_city_zip_codes = [
    19123,
    19102,
    19103,
    19106,
    19107,
    19109,
    19130,
    19146,
    19147,
]

In [ ]:
in_center_city = philly_tidy["RegionName"].isin(greater_center_city_zip_codes)
center_city = philly_tidy.loc[in_center_city].copy()
outside_center_city = philly_tidy.loc[~in_center_city].copy()

print(center_city['RegionName'].nunique(), outside_center_city['RegionName'].nunique())

## 5. Compare home value appreciation in Philadelphia

In [ ]:
def calculate_percent_increase(group_df):
    """
    Calculate the percent increase from 2020-03-31 to 2022-03-31.

    Note that `group_df` is the DataFrame for each group.
    """
    start = group_df.loc[group_df["Date"] == "2020-03-31", "ZHVI"].squeeze()
    end = group_df.loc[group_df["Date"] == "2022-03-31", "ZHVI"].squeeze()

    return (end - start) / start * 100

In [ ]:
center_city_change = center_city.groupby("RegionName").apply(calculate_percent_increase)
outside_change = outside_center_city.groupby("RegionName").apply(calculate_percent_increase)

center_city_avg = center_city_change.mean()
outside_avg = outside_change.mean()

print(f"Center City: {center_city_avg:.1f}%")
print(f"Outside Center City: {outside_avg:.1f}%")